# Week 6 Lab: Distillation — The Third Compression

**ECBS5200 — Practical Deep Learning Engineering**

Six weeks ago you trained a 149M ModernBERT on 113-class long-tail consumer
complaints and got macro F1 ≈ 0.21. Then you tried bigger models. Then you
tried a kitchen-sink recipe. Then you opened the data pipeline and learned
that what you'd been calling "the ceiling" was partly a data-composition
effect, not a model-scale ceiling. The semester's revised major finding,
from a paired bootstrap across the full canonical val set:

> **Scale beyond ~395M parameters does not produce a statistically
> significant macro F1 advantage when training data composition is held
> constant.** Most of the lift we initially attributed to scale was the
> training-data composition shift (train→train+test).

Week 6 closes the compression trilogy:

- **Week 5 part 1**: label compression (the merge mapping + MIN_CLASS_COUNT
  filter that gave us 113 classes)
- **Week 5 part 2**: weight compression (int8 / int4 quantization)
- **Week 6**: knowledge compression — distillation

> **The thesis to test today.** Distillation is the third compression.
> Like the other two, it can't recover what the data didn't have for
> *capacity*. But the question is whether it can transfer something else.

**Time budget:** ~80 minutes.

**How to use this notebook:**
- **GUIDED** cells run as-is. Read the prose; the cell does the work.
- **INTERACTIVE** cells require you to fill in values or write answers.
- `___` is a placeholder that will cause a `NameError` if not filled in.
- Several sections ask you to **PREDICT** before observing — write down
  your prediction first, then run the cell, then read the reveal. The memo
  rubric rewards specific predictions and specific corrections.

**Three acts today:**

- **Act 1 (~20 min)**: meet the teacher and look at what it actually does
- **Act 2 (~30 min)**: implement the KD loss; compare distilled vs vanilla
  student per tier
- **Act 3 (~30 min)**: capstone synthesis + memo prompts

## Kaggle setup (do this first!)

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → GPU T4
4. **Secrets** → add `HF_TOKEN` (HuggingFace read token)

This lab does not require write access — every artifact you need is
already on the Hub. A read token is sufficient.

In [ ]:
# GUIDED: environment setup. Run as-is.
import os, sys, subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# HuggingFace config — set BEFORE importing HF libraries.
if os.path.exists("/kaggle/working"):
    os.environ["HF_HOME"] = "/kaggle/working/.hf_cache"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# No --upgrade — replacing live modules causes circular import crashes
# (see Week 1 setup guide).
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.53", "accelerate>=0.34",
    "datasets", "scikit-learn", "matplotlib", "pandas",
])

# Defensive: peft (loaded transitively by some HF tooling) has a strict
# torchao>=0.16 check that raises if a stale torchao is present, even
# though we don't use it. Uninstall it cleanly.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
               check=False)
print("Packages installed.")

In [ ]:
# GUIDED: HuggingFace auth. Run as-is.
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: No HF_TOKEN found. Public reads will work but rate limits hit fast.")

In [ ]:
# GUIDED: clone the course repo (for utils/data_utils.py + canonical splits).
from pathlib import Path

if Path("/kaggle/working").exists():
    REPO_DIR = Path("/kaggle/working/applied-deep-learning")
elif Path("/content").exists():
    REPO_DIR = Path("/content/applied-deep-learning")
else:
    REPO_DIR = Path("./applied-deep-learning")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/earino/applied-deep-learning.git",
                           str(REPO_DIR)])
sys.path.insert(0, str(REPO_DIR))
print(f"Repo at: {REPO_DIR}")

In [ ]:
# GUIDED: imports.
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import Counter
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score, accuracy_score
from datasets import concatenate_datasets

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Hardware: T4 is recommended (matches the rest of the course), but the
# lab does analysis over precomputed logits + small KD-loss sanity
# checks on tiny random tensors. CPU works too — slower, fine for the
# lab. We just want a clean fallback path.
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    cc_major = torch.cuda.get_device_properties(0).major
    USE_BF16 = cc_major >= 8  # T4 is 7.5 → fp16. Ampere+ → bf16.
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
else:
    DEVICE = "cpu"
    AMP_DTYPE = torch.float32
    print("Device: CPU (no GPU detected). Kaggle T4 is the recommended path "
          "for this course, but the lab itself only needs CPU.")
print(f"AMP dtype: {AMP_DTYPE}")

## Where we are in the semester

Three compression mechanisms, one ceiling — that's the through-line of
Weeks 5–6. Two of them you've already measured:

1. **Label compression (Week 5 part 1)**: the merge map collapses 153 raw
   issues to 113 canonical classes. The MIN_CLASS_COUNT=5 filter drops
   everything below that. Outcome: a tractable label space at the cost of
   fragility on tail classes.

2. **Weight compression (Week 5 part 2)**: int8 / int4 quantization. The
   headline pattern: head accuracy survived; tail accuracy was where the
   fragility showed up. The data ceiling we'd been hitting all semester
   was visible in how compression *redistributed* errors across tiers.

3. **Knowledge compression (today)**: distillation. A 149M student tries
   to learn from a 32B teacher.

**Today's question is sharper than "does distillation help?".** It is:
*what does distillation actually transfer, and what's it limited by?*
Today you measure the answer per tier, in real numbers, on the same
val set you've used since Week 1.

---

# Act 1 — The teacher and what it actually does (~20 min)

A Qwen3-32B teacher exists. The base model is 32B parameters, but it was
adapted with **LoRA (rank 16)** — so only ~3.2M parameters were actually
updated during training (about 0.01% of the model). The remaining 32B stay
frozen. When you read "32B teacher" below, the right mental model is "32B
of frozen pretraining knowledge plus a small LoRA adapter trained on this
task." It was trained on train + test combined (79,278 examples), then
temperature-scaled. Its val numbers, from `teacher_card.json`:

| Metric | Value |
|---|---|
| Macro F1 | 0.322 |
| NLL | 1.256 |
| ECE | 0.021 |
| Head F1 (top-20 classes, n=5,155) | 0.652 |
| Mid F1 (rank 20–60, n=1,065) | 0.450 |
| **Tail F1 (rank 60–113, n=210)** | **?** |

Before we look at the tail number, **stop and predict**. Then we'll inspect
the teacher's actual probability distributions on a few held-out examples.

### 1a — Anchor: how much data does each tier have?

Before predicting the teacher's tail F1, look at how much data the
teacher had to learn from. These counts come from the data the teacher
was trained on (train + test combined, 79,278 examples). Per-class
training frequency is the ceiling that bounds per-class accuracy at
every model scale, including 32B.

In [ ]:
# GUIDED: load the canonical training data and count examples per class.
from utils.data_utils import load_course_data
_train_ds, _val_ds, _test_ds = load_course_data()
_combined_labels = list(_train_ds["label"]) + list(_test_ds["label"])
_label_counts = Counter(_combined_labels)
_sorted_classes = sorted(_label_counts.items(), key=lambda x: -x[1])

print(f"{'Rank':<6}{'Class':<8}{'# train+test ex.':>20}")
print("-" * 34)
print("HEAD (top 5):")
for i, (cls, cnt) in enumerate(_sorted_classes[:5], start=1):
    print(f"  #{i:<4}{cls:<6}{cnt:>20d}")
print("MID (around rank 30):")
for i, (cls, cnt) in enumerate(_sorted_classes[27:32], start=28):
    print(f"  #{i:<4}{cls:<6}{cnt:>20d}")
print("TAIL (rarest 5):")
n_classes = len(_sorted_classes)
for i, (cls, cnt) in enumerate(_sorted_classes[-5:], start=n_classes - 4):
    print(f"  #{i:<4}{cls:<6}{cnt:>20d}")

print(f"\nTotal classes: {n_classes}")
# Tail tier = ranks 60–113 (zero-indexed slice 59:113)
print(f"Median tail support (ranks 60–113): "
      f"{int(np.median([c for _, c in _sorted_classes[59:]]))} examples per class")

**Pause and read the numbers.** The tail's median support is single-digit
training examples per class. The head's top class has thousands.

That ratio — *how many examples per class the teacher saw in training* —
is the data ceiling, made concrete. The teacher's tail F1 isn't bounded
by its 32B parameters. It's bounded by how few examples per tail class
even the teacher had to learn from.

Use this when you make the next prediction.

### 1b — PREDICT before observing

The teacher is a 32B-parameter model. It was fine-tuned with LoRA on the
largest training set available (train + test, 79,278 examples), with a
careful three-stage cRT recipe and post-hoc temperature scaling. Its
*overall* macro F1 is 0.322 — the highest any model has reached on this
task across the entire semester.

**What's its tail F1?** (Tail = the 53 rarest classes, ranks 60–113. Total
of 210 val examples across these 53 classes.)

YOUR PREDICTION:

- Tail F1 (3-decimal place):
- Reasoning (one sentence):

In [ ]:
# GUIDED: download the teacher's val logits and look at the actual numbers.
# Logits are precomputed and hosted as a public dataset — no need to load the
# 32B teacher itself. The file holds the temperature-scaled val logits plus
# the precomputed labels and tier assignments.
TEACHER_REPO = "earino/ecbs5200-week6-teacher-logits"

teacher_logits_path = hf_hub_download(
    repo_id=TEACHER_REPO, repo_type="dataset",
    filename="val_logits_canonical_final.npz",
)
teacher_npz = np.load(teacher_logits_path)
teacher_val_logits = teacher_npz["logits"].astype(np.float32)
teacher_val_labels = teacher_npz["labels"]
teacher_val_tiers  = teacher_npz["tiers"]
teacher_val_preds  = teacher_val_logits.argmax(axis=-1)

print(f"Teacher val logits shape: {teacher_val_logits.shape}")
print(f"Tier distribution: {dict(Counter(teacher_val_tiers.tolist()))}")

In [ ]:
# INTERACTIVE: compute the teacher's per-tier macro F1.
# Hint: f1_score(labels[mask], preds[mask], labels=tier_label_set, average='macro').
# The tier_label_set is the list of class IDs that belong to that tier.
HEAD_LABELS = sorted(set(teacher_val_labels[teacher_val_tiers == "head"]))
MID_LABELS  = sorted(set(teacher_val_labels[teacher_val_tiers == "mid"]))
TAIL_LABELS = sorted(set(teacher_val_labels[teacher_val_tiers == "tail"]))

teacher_per_tier_f1 = {}
for tier_name, tier_label_set in [("head", HEAD_LABELS),
                                   ("mid",  MID_LABELS),
                                   ("tail", TAIL_LABELS)]:
    mask = teacher_val_tiers == tier_name
    teacher_per_tier_f1[tier_name] = ___  # FILL IN: per-tier macro F1
    print(f"  teacher {tier_name:5s} F1: {teacher_per_tier_f1[tier_name]:.3f}  (n={mask.sum()})")

### Compare your prediction to the actual tail F1

Don't skip the comparison. Write it down:

- Your predicted tail F1:
- Actual tail F1:
- Your error:

<details>
<summary><b>Click after you've written your reconciliation</b></summary>

The actual tail F1 is **~0.20** (0.198 to three decimals). A 32B-parameter,
carefully-fine-tuned teacher gets *0.198 macro F1* on the long tail of this
113-class task — barely above what a coin flip would give you on a balanced
53-way subproblem (~0.019), but the head and mid tiers carry the headline 0.322.

**Why does this matter pedagogically?** Because the casual claim "scale
beats long-tail" — which the field has been making in various forms since
at least 2020 — is not what we observe here. A 200× scale-up over our
original 149M baseline buys ~0.05 macro F1 lift on the tail. The data
ceiling Kandpal et al. 2023 documented at the LLM scale (factual recall
requires "many orders of magnitude" more parameters for rare facts)
applies here too at the supervised classification scale.

Hold this number in your head — 0.198 — for the rest of the lab. It's a
warning sign for what distillation can plausibly transfer: KD should not
be expected to *systematically* transfer tail decision quality that the
teacher itself does not reliably possess. A student can sometimes beat
its teacher on a metric — better inductive bias, the CE term anchoring
to hard labels, regularization effects, the teacher being locally
miscalibrated — so a small overshoot wouldn't be impossible, but a
large tail F1 lift would be surprising and would need a mechanism.

</details>

### 1c — What does the teacher's *probability distribution* look like?

Macro F1 is the argmax view. Distillation operates on the full softmax
distribution — the "dark knowledge" that hard labels can't carry. So
let's look at the teacher's actual probabilities on a few examples.

In [ ]:
# GUIDED: pick 3 val examples — one from each tier — and show the teacher's top-5 probabilities.
def softmax(logits):
    shifted = logits - logits.max(axis=-1, keepdims=True)
    e = np.exp(shifted)
    return e / e.sum(axis=-1, keepdims=True)

teacher_val_probs = softmax(teacher_val_logits)

# Pick one example from each tier (the first one we find).
example_indices = {}
for tier in ["head", "mid", "tail"]:
    idx = np.where(teacher_val_tiers == tier)[0][0]
    example_indices[tier] = int(idx)

for tier, idx in example_indices.items():
    true_label = int(teacher_val_labels[idx])
    probs = teacher_val_probs[idx]
    top5 = np.argsort(-probs)[:5]
    print(f"\n{tier.upper()} example (idx={idx}, true class={true_label}):")
    print(f"  teacher top-5: ", end="")
    print(", ".join(f"class {int(c)}={probs[c]:.3f}" for c in top5))
    print(f"  teacher's max probability: {probs.max():.3f}")
    print(f"  teacher's entropy:         {-(probs * np.log(probs.clip(min=1e-12))).sum():.3f}")

### 1d — PREDICT before reading the reveal

Look at the three examples above. Notice the entropy values. Now predict:

**Which tier do you expect to have the most peaked teacher distribution
(lowest entropy)?**

YOUR PREDICTION:

- Most peaked (lowest entropy):
- Reasoning (one sentence):

Now look at the three printed entropies. Reconcile **before opening the
reveal**:

- Your pick:
- Tier with the actually-lowest entropy in the printed examples:
- Were you right? If not, what was your wrong assumption?

<details>
<summary><b>Click after writing your reconciliation</b></summary>

In the typical case, **head** examples produce the most peaked teacher
distributions. The teacher has seen many examples of head classes during
training, has built confident decision boundaries around them, and
concentrates probability mass on a single class. Mid and tail examples
tend to show more spread — the teacher knows roughly what kind of
complaint it is but is less certain about which specific subclass.

**Why this matters for distillation:** the *spread* on mid/tail examples
is exactly the dark knowledge that hard labels cannot carry. A hard
one-hot label tells the student "this is class 47." A teacher's softmax
tells the student "this is probably class 47, but there's a real
12% chance it's class 53 and a 7% chance it's class 89, and these are
the relationships between the classes." If your tier showed something
different from this typical pattern (the entropy values can vary
example-by-example), think about why: maybe that specific head example
was genuinely ambiguous, or that tail example happens to be a clear case.

</details>

### 1e — Teacher calibration

The teacher's ECE on val (post-temperature-scaling, 15 equal-width bins) is
0.021. Below ~0.05 is "reasonably calibrated" by the standard the field
uses. We're going to ask whether this calibration property *transfers* to a
149M student through distillation. Hold the number — 0.021 — in mind.

In [ ]:
# GUIDED: confirm the teacher's ECE on val by recomputing it.
def expected_calibration_error(probs, labels, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    confs = probs.max(axis=-1)
    preds = probs.argmax(axis=-1)
    correct = (preds == labels)
    n = len(labels)
    ece = 0.0
    for i in range(n_bins):
        mask = (confs > edges[i]) & (confs <= edges[i + 1])
        if mask.sum() > 0:
            ece += (mask.sum() / n) * abs(correct[mask].mean() - confs[mask].mean())
    return float(ece)

teacher_ece = expected_calibration_error(teacher_val_probs, teacher_val_labels)
teacher_nll = float(-np.log(
    teacher_val_probs[np.arange(len(teacher_val_labels)), teacher_val_labels].clip(min=1e-12)
).mean())
print(f"Teacher ECE on val: {teacher_ece:.4f}")
print(f"Teacher NLL on val: {teacher_nll:.4f}")

---

# Act 2 — Distillation in practice (~30 min)

You will:

1. Implement the KD loss (the math you need to know going forward).
2. Run a short demo training to see the loss work.
3. Pull two pre-trained 149M ModernBERT students from the Hub:
   - `earino/ecbs5200-week6-vanilla-baseline` — trained with plain CE
   - `earino/ecbs5200-week6-distilled-student` — trained with KD against
     the 32B teacher (T_d=4, α=0.7, 3 epochs, identical seed and recipe)
   Same model, same data, same hyperparameters. Only the loss function
   differs. This is the cleanest possible isolation of "what does
   distillation specifically transfer."
4. Compare them per tier. Two predict-then-observe moments here. Each
   is a load-bearing reveal — write your predictions before running.

### 2a — Implement the KD loss

The standard knowledge distillation loss (Hinton et al. 2015) is a
weighted combination of two terms:

$$
\mathcal{L}_{\text{KD}} =
    \alpha \cdot T_d^2 \cdot \text{KL}\!\left(
        \sigma\!\left(\tfrac{t}{T_d}\right)
        \,\Big\|\,
        \sigma\!\left(\tfrac{s}{T_d}\right)
    \right)
  + (1-\alpha) \cdot \text{CE}(s,\,y_{\text{hard}})
$$

where:
- `s` and `t` are the student and teacher logits (not probabilities).
- `σ(·/T_d)` is softmax applied to logits divided by the temperature `T_d`,
  producing soft probability distributions over the 113 classes.
- The KL is **from the teacher distribution to the student distribution** —
  we're trying to match the student to the teacher, so the teacher is the
  reference. PyTorch's `F.kl_div(log_softmax(s/T), softmax(t/T))` computes
  exactly this direction (its first argument is interpreted as log-probs of
  the *approximation*, the second as the *target*).
- `T_d` is the **distillation temperature** — softens both distributions
  before computing KL.
- `α` is the weight on the soft-target KL term; (1−α) is the weight on
  the hard-label CE term.
- The `T_d²` factor cancels the `1/T_d²` gradient scaling that softmax/T
  introduces — without it, you'd have to scale the LR by `T_d²` to get
  comparable gradient magnitudes.

Today we use **T_d = 4, α = 0.7** (default values from the build prompt).
In the homework you'll sweep these.

In [ ]:
# INTERACTIVE: implement the KD loss.
#
# Use F.log_softmax on the student side, F.softmax on the teacher side.
# F.kl_div in PyTorch expects log-probs from one model and probs from the
# other; the convention is: F.kl_div(log_softmax(s/T), softmax(t/T), reduction='batchmean').
#
# Don't forget the T_d**2 multiplier on the KL term.

T_D = 4.0
ALPHA = 0.7

def kd_loss_fn(student_logits, teacher_logits, hard_labels, T_d=T_D, alpha=ALPHA):
    """Compute the KD loss: α·KL(σ(t/T) || σ(s/T))·T² + (1−α)·CE(s, hard_labels).

    Args:
        student_logits: tensor of shape (batch, num_classes).
        teacher_logits: tensor of shape (batch, num_classes), no grad.
        hard_labels:    tensor of shape (batch,) with class indices.

    Returns:
        scalar loss tensor (the weighted sum).
    """
    # FILL IN: compute the KL term.
    kl_term = ___
    # FILL IN: compute the CE term against the hard labels.
    ce_term = ___
    # FILL IN: combine them with α / (1−α) weighting.
    return ___

# Quick sanity check — should produce a positive scalar without NaN.
torch.manual_seed(SEED)
demo_s = torch.randn(8, 113, requires_grad=True)
demo_t = torch.randn(8, 113)
demo_y = torch.randint(0, 113, (8,))
demo_loss = kd_loss_fn(demo_s, demo_t, demo_y)
print(f"KD loss on random tensors: {demo_loss.item():.4f}")
assert not torch.isnan(demo_loss).item(), "KD loss is NaN — check your implementation."
assert demo_loss.item() > 0, "KD loss should be positive."

### 2b — BUG HUNT: a colleague's KD loss

A colleague sent you their KD loss implementation. They report:
*"It trains and the loss decreases. But the student isn't picking up
the teacher's calibration the way I expected."*

Below is their function. **It contains three subtle bugs.** Each bug
is silent — the function runs, returns a positive number, and the
loss decreases during training. None will crash. But each one
changes what the student actually learns.

Vague hints (no answers):

1. Two of the bugs are inside the KL term. One is in how the soft
   targets are formed; the other is in how the term is scaled.
2. One bug is in how the two terms are combined. If you wrote your
   `kd_loss_fn` correctly above, comparing line-by-line will surface it.

Find the three bugs **without running training**. Read carefully. Then
write a corrected version below.

**Time cap (read this before starting):** budget about 12 minutes for
the bug hunt. If you're stuck past 12 minutes, open the reveal *for the
first bug only*, internalize what to look for, and continue searching
for the other two on your own. Don't sink 25 minutes into this — the
memo cares about the *kind* of bug-hunting you do, not whether you
found all three unaided. Act 3 needs your attention more than the
third bug does.

YOUR ANSWER:
- Bug 1 (location + what's wrong):
- Bug 2 (location + what's wrong):
- Bug 3 (location + what's wrong):

In [ ]:
# DO NOT EDIT this function — it's the buggy version you're hunting.
def colleague_kd_loss(student_logits, teacher_logits, hard_labels, T_d=T_D, alpha=ALPHA):
    """Compute KD loss: α·KL(σ(t/T) || σ(s/T))·T² + (1−α)·CE(s, hard_labels). [BUGGY]"""
    kl_term = F.kl_div(
        F.softmax(student_logits / T_d, dim=-1),
        F.softmax(teacher_logits / T_d, dim=-1),
        reduction="batchmean",
    )
    ce_term = F.cross_entropy(student_logits, hard_labels)
    return alpha * ce_term + (1 - alpha) * kl_term

# Run the colleague's version on the SAME random tensors. It returns a number.
# The number is wrong, but nothing crashes — that's why this bug is hard to find.
torch.manual_seed(SEED)
_demo_s = torch.randn(8, 113, requires_grad=True)
_demo_t = torch.randn(8, 113)
_demo_y = torch.randint(0, 113, (8,))
print(f"colleague_kd_loss on the same batch: "
      f"{colleague_kd_loss(_demo_s, _demo_t, _demo_y).item():.4f}")
print(f"your kd_loss_fn on the same batch:   "
      f"{kd_loss_fn(_demo_s, _demo_t, _demo_y).item():.4f}")
print("Different numbers. The colleague's value is wrong; find why.")

In [ ]:
# INTERACTIVE: write a corrected version. Don't just copy your kd_loss_fn —
# write `fixed_kd_loss` so the diff vs `colleague_kd_loss` makes the bugs
# explicit. (The expected fixed value should match your kd_loss_fn output.)

def fixed_kd_loss(student_logits, teacher_logits, hard_labels, T_d=T_D, alpha=ALPHA):
    """Corrected version of colleague_kd_loss."""
    # FILL IN: write the corrected function. Keep the same overall shape
    # (one KL term + one CE term + α weighting); fix only the bugs.
    return ___

# Verify your fix matches your earlier kd_loss_fn on the same batch.
_fixed = fixed_kd_loss(_demo_s, _demo_t, _demo_y).item()
_yours = kd_loss_fn(_demo_s, _demo_t, _demo_y).item()
assert abs(_fixed - _yours) < 1e-5, \
    f"fixed_kd_loss disagrees with your kd_loss_fn ({_fixed:.4f} vs {_yours:.4f})."
print(f"fixed_kd_loss matches your kd_loss_fn: {_fixed:.4f}")

<details>
<summary><b>Click after writing all three bugs and the fixed function</b></summary>

**Bug 1 — `F.softmax` on the student input to `F.kl_div`.**
`F.kl_div(input, target)` expects `input` to be **log**-probabilities.
The colleague used `F.softmax`, so they passed probabilities where
log-probabilities were required. Numerically the call doesn't crash, but
the gradient computed inside KL is wrong. The fix is `F.log_softmax`.

**Bug 2 — missing `* (T_d ** 2)` on the KL term.**
Softening logits by T_d shrinks gradient magnitudes by 1/T_d². Without
the T_d² rescaling, the KL term's gradient contribution is too small
relative to CE — effectively underweighting the soft targets. Hinton et
al. 2015 introduced the T² multiplier exactly to neutralize this.

**Bug 3 — α and (1−α) are swapped.**
The colleague wrote `alpha * ce_term + (1 - alpha) * kl_term`, which
weights CE by α and KL by 1−α. With α=0.7, that means **CE gets 70%
weight and KL gets 30%** — the opposite of the intended recipe.
Calibration transfer (which lives in the KL term) is dampened by 2.3×.
That matches the colleague's symptom: "isn't picking up calibration."

All three bugs survive a "loss decreases during training" check, because
any reasonable loss with at least some teacher signal will trend down.
This is why **gradient-checking and value-comparison against a known-
good reference** matter more than just "loss goes down."

</details>

### 2c — KD vs vanilla CE on the same batch

Same random batch you just used for the sanity check. Compute (a) the
vanilla CE loss a no-distillation student would see on this batch, and
(b) the KD loss your implementation produces. Then look at the **KL
component on its own** — that's the dense-target information channel
hard-label CE doesn't have. (Both losses produce gradients on every
logit; the difference is what *target* those gradients are pulled
toward — a one-hot vector for CE versus a 113-class soft distribution
for KD.)

We're skipping a full 3-epoch training run today. The two checkpoints
you'll download in §2c are the result of that. Their training loop is
the same one you'd write here, just longer (and with a real model).

In [ ]:
# GUIDED: same random batch from the sanity check, two losses + the KL component.
demo_ce_only = F.cross_entropy(demo_s, demo_y)
demo_kd_full = kd_loss_fn(demo_s, demo_t, demo_y)
demo_kl_only = F.kl_div(
    F.log_softmax(demo_s / T_D, dim=-1),
    F.softmax(demo_t / T_D, dim=-1),
    reduction="batchmean",
) * (T_D ** 2)

print(f"  Vanilla CE loss on this batch:    {demo_ce_only.item():.4f}")
print(f"  KD loss (KL + CE) on this batch:  {demo_kd_full.item():.4f}")
print(f"  KL term alone (T_d²-scaled):      {demo_kl_only.item():.4f}")
print()
print("  KL gives the student a dense target distribution over all 113")
print("  classes; hard-label CE gives a one-hot target. Both produce")
print("  gradients on every logit (softmax-CE's gradient is p_j − 1[j=y]")
print("  on every j), but only KL's target encodes *how* mass should be")
print("  distributed across the non-true classes — the 'dark knowledge'")
print("  hard labels can't carry.")

### 2d — Download the full distilled and vanilla checkpoints from Hub

Both 149M ModernBERT students were trained on the same data (train+test
combined) with the same hyperparameters, the same random seed, and the
same number of epochs. The *only* difference is the loss function:

| Student | Loss |
|---|---|
| `earino/ecbs5200-week6-vanilla-baseline` | plain CE on hard labels |
| `earino/ecbs5200-week6-distilled-student` | α·KL(t/T_d ‖ s/T_d)·T_d² + (1−α)·CE, T_d=4, α=0.7 |

> ### ⚠️ ACADEMIC SETUP — NEVER DO THIS IN PRODUCTION
>
> Both vanilla and distilled students were trained on **train + test
> combined (79,278 examples)** — the same composition the teacher saw.
> **Repurposing the test split as training data is acceptable here ONLY
> because this is a classroom dataset with three labeled splits and we
> want the strongest possible teacher.** In any real project with a
> true held-out test set you reserve for final evaluation, this would
> be a fatal mistake — your reported metrics on that test set would
> overstate generalization.
>
> The **val split remains untouched** all term, so vanilla vs distilled
> comparisons are still valid *relative to each other*. We make the
> comparison vs the train-only Week 1 baseline (0.209) explicit in §3a
> so distillation doesn't get credit for the data shift. The shift is
> the "data confound" the term arc closes on.

Both vanilla and distilled trained on train+test → their vanilla macro
F1 (~0.264) is higher than the train-only Week 1 baseline (0.209).
Vanilla-vs-distilled below isolates the loss function, not the data.

We don't need to reload the full models — both Hub repos include a
`val_predictions.npz` file that contains each model's raw val logits,
argmax predictions, true labels, and tier assignments. Loading those is
instant and gives us everything we need for analysis.

In [ ]:
# GUIDED: download both checkpoints' val_predictions.npz.
VANILLA_REPO   = "earino/ecbs5200-week6-vanilla-baseline"
DISTILLED_REPO = "earino/ecbs5200-week6-distilled-student"

vanilla_path = hf_hub_download(
    repo_id=VANILLA_REPO, repo_type="model",
    filename="val_predictions.npz",
)
distilled_path = hf_hub_download(
    repo_id=DISTILLED_REPO, repo_type="model",
    filename="val_predictions.npz",
)

vanilla_npz   = np.load(vanilla_path)
distilled_npz = np.load(distilled_path)

vanilla_logits   = vanilla_npz["logits"].astype(np.float32)
vanilla_preds    = vanilla_npz["preds"]
vanilla_labels   = vanilla_npz["labels"]
val_tiers        = vanilla_npz["val_tiers"]

distilled_logits = distilled_npz["logits"].astype(np.float32)
distilled_preds  = distilled_npz["preds"]
# (Labels and tiers are identical across both — same val set.)

# Sanity: same val examples in same order.
assert (distilled_npz["labels"] == vanilla_labels).all()
assert (distilled_npz["val_tiers"] == val_tiers).all()
print(f"Vanilla logits:   shape {vanilla_logits.shape}")
print(f"Distilled logits: shape {distilled_logits.shape}")

### 2e — Compute per-tier metrics for both students

We want four numbers per student per tier: macro F1, accuracy, ECE, NLL.
Plus the same metrics aggregated overall. The structure is exactly what
you'd report in a deployment-decision document.

In [ ]:
# INTERACTIVE: write the per-tier metrics function.
# Per-tier F1 you already wrote in Act 1 — that part is filled in for
# you here. The new piece is **per-tier ECE**: take the same ECE
# function you computed at the corpus level, and apply it within each
# tier subset. This is the cell that lets you ask "did calibration
# improve uniformly, or concentrate on a specific tier?"

def compute_metrics(logits, labels, tiers):
    """Return dict with overall + per-tier macro F1 and ECE, plus accuracy and NLL."""
    probs = softmax(logits)
    preds = probs.argmax(axis=-1)

    # Overall metrics
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    ece = expected_calibration_error(probs, labels)
    nll = -np.log(probs[np.arange(len(labels)), labels].clip(min=1e-12)).mean()

    # Per-tier macro F1 (auto-filled — same pattern as Act 1)
    per_tier_f1 = {}
    for tier_name, tier_label_set in [("head", HEAD_LABELS),
                                       ("mid",  MID_LABELS),
                                       ("tail", TAIL_LABELS)]:
        mask = tiers == tier_name
        per_tier_f1[tier_name] = f1_score(
            labels[mask], preds[mask],
            labels=tier_label_set, average="macro", zero_division=0,
        )

    # Per-tier ECE (FILL IN).
    # Reuse `expected_calibration_error`, but pass only the rows for
    # the current tier (use the mask). What does it mean for calibration
    # to be reported per-tier vs. globally? Think about that as you
    # write the line.
    per_tier_ece_dict = {}
    for tier_name in ["head", "mid", "tail"]:
        mask = tiers == tier_name
        per_tier_ece_dict[tier_name] = ___  # FILL IN: per-tier ECE
    return {
        "macro_f1": float(macro_f1), "accuracy": float(acc),
        "ece": float(ece), "nll": float(nll),
        "per_tier": per_tier_f1,
        "per_tier_ece": per_tier_ece_dict,
    }

vanilla_metrics   = compute_metrics(vanilla_logits,   vanilla_labels, val_tiers)
distilled_metrics = compute_metrics(distilled_logits, vanilla_labels, val_tiers)

print("Vanilla baseline:")
print(f"  macro F1: {vanilla_metrics['macro_f1']:.4f}   "
      f"accuracy: {vanilla_metrics['accuracy']:.4f}   "
      f"ECE: {vanilla_metrics['ece']:.4f}   "
      f"NLL: {vanilla_metrics['nll']:.4f}")
print(f"  head: {vanilla_metrics['per_tier']['head']:.4f}   "
      f"mid: {vanilla_metrics['per_tier']['mid']:.4f}   "
      f"tail: {vanilla_metrics['per_tier']['tail']:.4f}")

print("\nDistilled student:")
print(f"  macro F1: {distilled_metrics['macro_f1']:.4f}   "
      f"accuracy: {distilled_metrics['accuracy']:.4f}   "
      f"ECE: {distilled_metrics['ece']:.4f}   "
      f"NLL: {distilled_metrics['nll']:.4f}")
print(f"  head: {distilled_metrics['per_tier']['head']:.4f}   "
      f"mid: {distilled_metrics['per_tier']['mid']:.4f}   "
      f"tail: {distilled_metrics['per_tier']['tail']:.4f}")

### 2f — PREDICT before observing #1: where does distillation's F1 lift land?

You now have two students, identical except for the loss function. The
distilled student saw teacher soft targets *in addition to* the same
hard labels the vanilla student saw. Where do you predict the macro F1
lift will land?

The four reasonable prior beliefs:

**(a) Uniform across tiers.** The teacher contributes information about
all classes equally, so the lift is spread evenly: head, mid, and tail
all lift by similar amounts.

**(b) Concentrated on head.** The teacher has the most signal where it
saw the most data; the lift concentrates on the data-rich tiers. Tail
stays flat because no amount of teacher-signal can compensate for the
student lacking tail examples.

**(c) Concentrated on tail.** The teacher's "dark knowledge" about
rare classes is exactly what the hard-label student is missing; the
distilled student picks up tail classes the vanilla student couldn't.

**(d) Tail-helping but mid-flat.** The teacher's soft targets help most
where the student's own signal is weakest; head and mid are saturated,
tail is the place to gain.

YOUR PREDICTION:

- Pick (a), (b), (c), or (d):
- Reasoning (one or two sentences):

In [ ]:
# GUIDED: print the per-tier comparison table.
print(f"{'Tier':<8} {'Vanilla F1':>12} {'Distilled F1':>14} {'Δ':>10}")
print("-" * 46)
for tier in ["head", "mid", "tail"]:
    v = vanilla_metrics["per_tier"][tier]
    d = distilled_metrics["per_tier"][tier]
    print(f"{tier:<8} {v:>12.4f} {d:>14.4f} {d - v:>+10.4f}")
print("-" * 46)
print(f"{'overall':<8} {vanilla_metrics['macro_f1']:>12.4f} "
      f"{distilled_metrics['macro_f1']:>14.4f} "
      f"{distilled_metrics['macro_f1'] - vanilla_metrics['macro_f1']:>+10.4f}")

### Compare your prediction to the actual pattern

Look at the table above. Which of (a)/(b)/(c)/(d) was right? Don't skip
the reconciliation. Write it down:

- Your pick:
- Actual pattern (in your own words, 1–2 sentences):
- Were you right? If not, what was your wrong assumption?

<details>
<summary><b>Click after writing your reconciliation</b></summary>

The actual pattern is **(b)**: head F1 lifts by about 0.02 — a real,
bootstrap-significant improvement. Mid has a similar point-estimate
lift (~0.02), but its 95% CI includes zero because of the smaller
sample size (n=1,065 mid examples vs n=5,155 head — so the bootstrap
CI is wider on mid even though the point estimate is comparable).
Tail lifts a small amount (~+0.005) and is also indistinguishable from
zero given n=210. Sign of the mid and tail point estimates flips
across reruns — that's the signature of a true null effect under fp16
run-to-run nondeterminism plus sparse per-class data, not evidence of
a stable small effect being washed out.

Why (b) and not (c) or (d)? **The teacher's tail F1 was 0.198.** It
can't transfer knowledge it doesn't have. The teacher hit the same data
ceiling on the tail that every model on this dataset has hit since
Week 1. Distillation is not a workaround for the data ceiling — it
can only transfer information the teacher has, and the teacher is data-
bound on the tail just like the student is.

This is the empirical core of "wanting #3 cracks here": you wanted
distillation to recover the tail. Three weeks ago you wanted big models
to do it. Five weeks ago you wanted clever loss tricks. None of them
crack the data ceiling. **In this scaffold, the train→train+test data
shift moved vanilla MBL-base macro F1 by about +0.05 (Week 1 baseline
0.209 → vanilla 0.264). That is several times larger than what
distilling from a 32B teacher on top of the same data buys.**

Hold this number too — Δ macro F1 ≈ +0.015 overall, with the head-tier
lift the only one clearly distinguishable from zero — because we're
about to look at the second axis of distillation transfer, where the
picture is very different.

</details>

### 2g — PREDICT before observing #2: does ECE follow the same per-tier pattern as F1?

You just observed that F1 lifts mostly on head, almost not at all on
mid+tail. Now: **does the calibration improvement (ECE) follow the same
pattern, or a different one?**

The two prior beliefs to choose between:

**(a) Same pattern as F1.** ECE will improve where F1 improved (head),
and mid+tail ECE will be similar to vanilla.

**(b) Different pattern from F1.** ECE could improve in tiers where F1
didn't move at all.

YOUR PREDICTION:

- Pick (a) or (b):
- What assumption about how the loss function uses confidence are you
  relying on?

In [ ]:
# GUIDED: show per-tier ECE using the values you computed in compute_metrics.
#
# Note on tail ECE: tail has n=210 across 15 ECE bins, so the per-bin
# population can be small (often < 20). Tail ECE here is a descriptive
# diagnostic, not a stable population estimate — read tier-level *trends*
# (does ECE drop on every tier?), not single-decimal precision.
print(f"{'Tier':<8} {'Vanilla ECE':>13} {'Distilled ECE':>15} {'Δ':>10}")
print("-" * 48)
for tier in ["head", "mid", "tail"]:
    v = vanilla_metrics["per_tier_ece"][tier]
    d = distilled_metrics["per_tier_ece"][tier]
    print(f"{tier:<8} {v:>13.4f} {d:>15.4f} {d - v:>+10.4f}")
print("-" * 48)
print(f"{'overall':<8} {vanilla_metrics['ece']:>13.4f} "
      f"{distilled_metrics['ece']:>15.4f} "
      f"{distilled_metrics['ece'] - vanilla_metrics['ece']:>+10.4f}")
print(f"\n         Vanilla NLL: {vanilla_metrics['nll']:.4f}")
print(f"         Distilled NLL: {distilled_metrics['nll']:.4f}")
print(f"         NLL Δ:         {distilled_metrics['nll'] - vanilla_metrics['nll']:+.4f}")

### Now build the mechanism — before opening the reveal

Look at the table you just printed. Then **write 2–3 sentences explaining
what about the loss function would produce a pattern like this.** Use the
words "hard label," "softmax," and "gradient." Don't open the reveal
until you've written it.

- Your pick (a or b):
- Your mechanism (2–3 sentences):
- Were you right? If not, what was your wrong assumption?

<details>
<summary><b>Click after writing your mechanism</b></summary>

The pattern is **(b)** — ECE improves on every tier, including the
tiers where F1 didn't move. Overall drop ~3×. NLL drops ~14%.

Compare your mechanism to this one: **CE has a one-hot target, so
non-true classes are not given differentiated target probabilities —
they're all just "not the answer." KL has a dense teacher target, so
class 12, class 53, and class 89 receive different target pressures
proportional to the teacher's belief about each.** Both losses produce
gradients on every logit (softmax-CE's gradient is p_j − 1[j=y] on
every j), but only KL's target shape carries information about *how*
probability mass should distribute across the non-true classes. That
means a tail example can teach the student the probability shape over
the other 112 classes even when the student doesn't have enough data
to learn the right argmax on that tier. Capacity transfer needs data
per class. Calibration transfer needs only that the loss can read the
teacher's probability mass — which is what hard labels cannot do.

Hold this as the lab's headline: **distillation has two transfer
axes, and the data ceiling binds them very differently.** Capacity
transfer (argmax/F1) needs per-class data — the same constraint
that's bounded every model on this dataset since Week 1. Calibration
transfer needs much less per-class data, because the KL loss can
read the teacher's *probability shape* even on classes the student
rarely sees. (Strictly speaking calibration is also data-bounded —
with zero examples for a class, there is no distribution to match —
but the per-class data threshold is far lower than for capacity.)
This is what you'll defend in the memo.

</details>

### 2h — Paired bootstrap to confirm the per-tier deltas

In [ ]:
# GUIDED: paired bootstrap on per-tier F1 deltas. Run as-is.
#
# This does 3,000 sklearn F1 calls (1,000 resamples × 3 tiers × 2 arms).
# Verified on Kaggle T4 instance: ~10–15 seconds total (sklearn runs on
# CPU regardless of the GPU type). The tqdm progress bar is there so you
# can see it's working, not because the cell is slow.
#
# **If this cell takes longer than ~60 seconds** (CPU contention, noisy
# Kaggle instance, slower runtime), drop `n_resamples` to 500. The CI
# will be slightly wider but the conclusions are unchanged at this
# sample size.
#
# (Why GUIDED, not INTERACTIVE: the resampling math is bookkeeping, not a
# load-bearing pedagogical moment. The interpretation of the CIs is the
# load-bearing part — we do that in the next cell.)

from tqdm.auto import tqdm

def paired_bootstrap_per_tier(preds_a, preds_b, labels, tiers, n_resamples=1000, seed=SEED):
    rng = np.random.RandomState(seed)
    out = {}
    for tier_name, tier_label_set in [("head", HEAD_LABELS),
                                       ("mid",  MID_LABELS),
                                       ("tail", TAIL_LABELS)]:
        idx = np.where(tiers == tier_name)[0]
        deltas = np.zeros(n_resamples)
        for r in tqdm(range(n_resamples), desc=f"bootstrap {tier_name}", leave=False):
            samp = rng.choice(idx, size=len(idx), replace=True)
            f1_a = f1_score(labels[samp], preds_a[samp],
                            labels=tier_label_set, average="macro", zero_division=0)
            f1_b = f1_score(labels[samp], preds_b[samp],
                            labels=tier_label_set, average="macro", zero_division=0)
            deltas[r] = f1_b - f1_a
        out[tier_name] = {
            "median": float(np.median(deltas)),
            "ci_low": float(np.quantile(deltas, 0.025)),
            "ci_high": float(np.quantile(deltas, 0.975)),
            "ci_excludes_zero": bool(np.quantile(deltas, 0.025) > 0
                                     or np.quantile(deltas, 0.975) < 0),
        }
    return out

print("Paired bootstrap (1000 resamples) on Δ(distilled − vanilla)...")
boot = paired_bootstrap_per_tier(vanilla_preds, distilled_preds, vanilla_labels, val_tiers)

print(f"\n{'Tier':<8} {'median Δ':>10} {'95% CI':>22} {'verdict':>20}")
print("-" * 64)
for tier in ["head", "mid", "tail"]:
    r = boot[tier]
    sig = "significant lift" if (r["ci_excludes_zero"] and r["median"] > 0) else "within noise"
    print(f"{tier:<8} {r['median']:>+10.4f}   [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]   {sig:>20}")

Note that "within noise" doesn't mean "no effect" — it means we cannot
distinguish the effect from zero given the sample size. Tail has only
210 val examples across 53 classes; even a real effect of +0.01 might
not be detectable. **For the memo, the right framing is "the CI
includes zero," not "the lift is zero."** This kind of distinction
is exactly what good empirical practice looks like. Don't over-claim;
don't under-claim.

---

# Act 3 — Capstone synthesis (~30 min)

Three compression mechanisms, one ceiling — that was the framing six
weeks ago. Now the picture is sharper:

- **Capacity** (the argmax F1 view) hits the data ceiling that has
  defined this dataset since Week 1. Distillation lifted only the head
  tier above noise.
- **Calibration** (the full-distribution view) needs less per-class
  data than capacity to transfer. ECE dropped ~3× across all tiers,
  even ones where F1 didn't move.

Distillation has two transfer axes, and the data ceiling binds them
very differently. That's the empirical conclusion of the lab. Now:
what does it mean for what you'd actually ship?

### 3a — The data confound (data >> distillation, at the macro level)

In [ ]:
# GUIDED: print the data-confound table.
print(f"{'Setup':<60} {'macro F1':>10}")
print("-" * 72)
print(f"{'MBL-base full FT, train-only, plain CE (Week 1 baseline)':<60} {0.209:>10.3f}")
print(f"{'MBL-base full FT, train+test, plain CE (Week 6 vanilla)':<60} "
      f"{vanilla_metrics['macro_f1']:>10.3f}")
print(f"{'MBL-base full FT, train+test, KD from 32B (Week 6 distilled)':<60} "
      f"{distilled_metrics['macro_f1']:>10.3f}")
print("-" * 72)
_data_lift = vanilla_metrics["macro_f1"] - 0.209
_distill_lift = distilled_metrics["macro_f1"] - vanilla_metrics["macro_f1"]
print(f"\nLift from 'see test data too':       {_data_lift:+.3f}")
print(f"Lift from 'also distill from 32B':   {_distill_lift:+.3f}")
print(f"\nRatio (data / distillation): {_data_lift / max(_distill_lift, 1e-9):.1f}× — "
      f"the data composition shift bought several times more macro F1 "
      f"than the 32B teacher did.")

Most of the macro F1 lift came from data composition, not distillation.
Hold the ratio. We come back to it in 3d.

### 3b — But calibration is a different story

In [ ]:
# GUIDED: print the calibration table.
print(f"{'Model':<40} {'ECE':>8} {'NLL':>8}")
print("-" * 58)
print(f"{'Teacher (Qwen3-32B post-temp-scale)':<40} {teacher_ece:>8.3f} {teacher_nll:>8.3f}")
print(f"{'Vanilla student (CE only)':<40} "
      f"{vanilla_metrics['ece']:>8.3f} {vanilla_metrics['nll']:>8.3f}")
print(f"{'Distilled student (KD)':<40} "
      f"{distilled_metrics['ece']:>8.3f} {distilled_metrics['nll']:>8.3f}")
print("-" * 58)
ratio = vanilla_metrics["ece"] / distilled_metrics["ece"]
print(f"\nKD's ECE improvement over vanilla: {ratio:.2f}× better calibration.")

### 3c — Threshold/coverage analysis: where calibration matters operationally

A common deployment pattern: the model auto-routes only when its top
probability exceeds a threshold. Below the threshold, the example
escalates to a human reviewer. **Whether that policy works at all
depends on whether the model's confidence values are *meaningful* —
which is exactly what calibration measures.**

You're going to compute, at thresholds 0.5, 0.6, 0.7, 0.8, 0.9, for
both vanilla and distilled:

- **Coverage:** fraction of val examples whose max-prob ≥ threshold
- **Accuracy@threshold:** accuracy on the covered subset only

A well-calibrated model has the property that *coverage and accuracy
move together as you raise the threshold*. A poorly-calibrated model
has examples that look confident but are wrong — so accuracy doesn't
rise the way coverage falls.

In [ ]:
# INTERACTIVE — FILL IN: complete the threshold/coverage table. **The §3d
# deployment decision and the homework §3c questions both need these exact
# numbers, so do NOT skip this fill-in.** The four reading questions below
# are deferred to homework, but the table itself must be populated now.
#
# Note on `n_covered`: at high thresholds the covered count drops fast —
# coverage 0.04 means only ~250 of 6,430 val examples are above threshold.
# Accuracy 1.0 on 8 covered examples is descriptive, not a population
# estimate. Always report n_covered alongside.
def coverage_table(probs, labels, thresholds=(0.5, 0.6, 0.7, 0.8, 0.9)):
    """For each threshold, return (threshold, coverage, accuracy_when_covered, n_covered)."""
    confs = probs.max(axis=-1)
    preds = probs.argmax(axis=-1)
    rows = []
    for t in thresholds:
        cov_mask = confs >= t
        n_covered = int(cov_mask.sum())
        coverage = float(cov_mask.mean())
        # FILL IN: accuracy = mean(preds[cov_mask] == labels[cov_mask]),
        # but handle the empty-mask case (no examples above threshold).
        accuracy = ___
        rows.append((t, coverage, accuracy, n_covered))
    return rows

vanilla_probs_arr   = softmax(vanilla_logits)
distilled_probs_arr = softmax(distilled_logits)

vanilla_cov   = coverage_table(vanilla_probs_arr,   vanilla_labels)
distilled_cov = coverage_table(distilled_probs_arr, vanilla_labels)

print(f"{'Thresh':<8}{'Van.cov':>9}{'Van.acc':>9}{'Van.n':>8}    "
      f"{'Dist.cov':>10}{'Dist.acc':>10}{'Dist.n':>8}")
print("-" * 72)
for (t, cov_v, acc_v, n_v), (_, cov_d, acc_d, n_d) in zip(vanilla_cov, distilled_cov):
    print(f"{t:<8.2f}{cov_v:>9.3f}{acc_v:>9.3f}{n_v:>8d}    "
          f"{cov_d:>10.3f}{acc_d:>10.3f}{n_d:>8d}")

### Read the table you just computed (one sentence in class — the rest
is in the homework)

In **one sentence**, before moving on: what is the operational
difference between a well-calibrated and a poorly-calibrated model
when the downstream system uses confidence thresholds at all?

- YOUR ANSWER:

*(The four detailed threshold/coverage questions — Vanilla coverage
@ 0.70, Distilled accuracy @ 0.70, the 95% SLA threshold for each
model, and the operational-difference summary — are in the homework.
You'll need them for the memo. Read your table now while it's fresh,
but don't burn class time writing them out — go straight to §3d.
**Make sure you've actually filled in and run the coverage_table cell
above — §3d's deployment artifact references those numbers.**)*

### 3d — Deployment decision (in-class artifact, due before leaving)

Pick ONE of the three scenarios below. Fill in **every** field with a
specific value from a table you've computed today (don't write "high"
or "low" — write the number). The rubric checks that your threshold
and coverage values came from your own §3c table.

**Form check** — an example of a single filled field's shape:
`Coverage at T: 0.500` (i.e. T=0.80 in your §3c table; not "high"
or "around half"). After you've filled in *your* fields, the reveal
at the bottom has a complete worked example for Scenario B to check
structure against.

**Scenario A — High-throughput routing.** Millions of complaints/day,
misroutes are expensive, latency < 50 ms, downstream uses argmax of
top class.

**Scenario B — Legal-discovery review.** Every classification reviewed
by a human; a confidence ≥ T should fast-track approval. False
confident wrongs create liability.

**Scenario C — Research dashboard (aggregate).** Aggregate totals over
millions of complaints; individual classifications don't matter, only
the per-class counts do.

**YOUR DEPLOYMENT DECISION (fill in every field):**

- Scenario chosen (A / B / C):
- Model chosen (vanilla / distilled / teacher):
- One-line justification (must reference at least one number from
  §3a, §3b, or §3c — name the section):

Threshold policy (skip only if you chose Scenario C, where thresholds
don't apply — but say so explicitly):

- Auto-route confidence threshold T:
- Coverage at T (from your table):
- Accuracy at T (from your table):
- Decision rule (one sentence using your T and the model name):

**One constraint that would flip your decision:**

- Constraint name:
- Threshold of constraint at which you'd switch:
- Alternative choice you'd switch to:

<details>
<summary><b>Click only after you've filled in all the fields above</b></summary>

There isn't one right scenario→model pairing. There IS a wrong shape
of answer: anything that doesn't reference numbers from your tables,
or anything that picks a model without naming the downstream property
it serves (argmax / threshold / aggregate).

A defensible answer pattern looks like:

> "Scenario B, distilled student, T=0.70 (vanilla acc@0.70 ≈ X%,
> distilled acc@0.70 ≈ Y%; only the distilled model meets SLA at
> usable coverage). Decision rule: auto-approve if max_prob ≥ 0.70,
> otherwise escalate to senior reviewer. Constraint that would flip:
> if T4 latency budget dropped below ~3 ms/example, I'd switch to a
> smaller distilled-then-quantized variant."

Notice the structure: scenario picks the property → property picks
the model → model + table picks the threshold → threshold + SLA picks
the decision rule → constraint names a specific switch point. That's
the shape every deployment dossier needs.

</details>

### 3e — Memo prompts (homework — finish before next class)

The Week 6 memo is organized around **one** central question, with five
sub-prompts that all feed it:

> **You've now seen three forms of compression — class merging, weight
> quantization, knowledge distillation — and one ceiling that all three
> failed to break. Defend your model choice for production: which
> compression(s) do you accept, which do you reject, and what's the one
> constraint under which your answer would change?**

The five prompts below all serve this central question. Your in-class
§3d artifact is the seed of Prompt 5; the rest you write at home.
The full memo is ~5–6 hours including the hyperparameter exploration
in the homework.

**Prompt 1 — What did distillation transfer?**

Per-tier F1 deltas (distilled − vanilla). Where did the lift land? What
does today's evidence — including the calibration-vs-capacity
decomposition — tell you about the *type* of information that got
transferred, not just the amount?

**Prompt 2 — What didn't distillation fix?**

The data ceiling shows up — tail F1 stays low for student, vanilla,
AND teacher. Defend: is this a property of the task, the data, or the
method? What additional measurements would you make to tell?

**Prompt 3 — Hyperparameters and the noise floor.**

(For the homework, you'll sweep T_d × α.) Looking at today's per-tier
bootstrap CIs, what's the *noise floor* on this measurement?
Specifically: how big does an effect have to be before you'd believe it
is real? Tie your answer to a specific tier and a specific CI.

**Prompt 4 — Three compressions, three lessons.**

Label compression (Week 5 part 1), weight compression (Week 5 part 2),
knowledge compression (Week 6). Which two of the three interact most
strongly on this dataset? Which two are nearly orthogonal? Defend with
a number from your measurements.

**Prompt 5 — Defend the deployment.**

Pick one of the three models from §3c (or a fourth: an even smaller
distilled model, an int8-quantized distilled model, something else
you'd argue for). Specify: what's the one constraint that would flip
your answer? Be specific — name the constraint, name the alternative
choice, name the threshold of the constraint at which you'd switch.

This prompt is the seed of the **final dossier** (the model-decision
document due before the final exam). Today's answer doesn't need to
be polished; it needs to be defensible.

---

### End of Week 6 lab

Save your notebook (it should now have your filled-in cells, your
predictions, your reconciliations, and your initial memo prompts).
Download it. The homework continues with hyperparameter exploration
and finishes the memo. You're 80% of the way to a deployment-defense
document — the last 20% is the homework.

**What you measured today, in one sentence:** distillation has two
transfer axes — capacity (tightly data-bounded) and calibration
(much less data-bounded at this scale) — and the long-tail ceiling on
this dataset binds the first far more than the second.

**What distillation did NOT do on this dataset:** raise tail F1 above
the teacher's own tail-data ceiling. **What it DID do:** improve
calibration across all tiers (ECE ~3×, NLL ~14%) and lift head F1
clearly above noise.